In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Set seed for reproducibility
tf.random.set_seed(42)

# -------------------------------
# 📥 Load and preprocess MNIST
# -------------------------------
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize input data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten images to vectors of size 784
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# One-hot encode labels
y_train_onehot = to_categorical(y_train, 10)
y_test_onehot = to_categorical(y_test, 10)

# -------------------------------
# 🧠 Model architecture (MLP)
# -------------------------------
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    return model


# -------------------------------
# 1️⃣ Custom training with tf.GradientTape
# -------------------------------
print("\n🔁 Training using tf.GradientTape")

# Create model
model_tape = create_model()

# Loss and optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Accuracy metric
train_accuracy = tf.keras.metrics.CategoricalAccuracy()

# Prepare dataset
batch_size = 64
epochs = 5
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train_onehot)).shuffle(1024).batch(batch_size)

# Training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            preds = model_tape(x_batch, training=True)
            loss = loss_fn(y_batch, preds)

        # Compute gradients and apply them
        grads = tape.gradient(loss, model_tape.trainable_weights)
        optimizer.apply_gradients(zip(grads, model_tape.trainable_weights))

        # Update accuracy metric
        train_accuracy.update_state(y_batch, preds)

        if step % 200 == 0:
            print(f"Step {step}, Loss: {loss.numpy():.4f}")

    # Epoch results
    epoch_acc = train_accuracy.result()
    print(f"Training Accuracy: {epoch_acc:.4f}")
    train_accuracy.reset_state()  # ✅ Corrected method name


# Evaluate on test set
test_preds = model_tape(x_test, training=False)
test_accuracy_metric = tf.keras.metrics.CategoricalAccuracy()
test_accuracy_metric.update_state(y_test_onehot, test_preds)
gradient_tape_accuracy = test_accuracy_metric.result().numpy()
print(f"\n✅ Test Accuracy (GradientTape): {gradient_tape_accuracy:.4f}")


# -------------------------------
# 2️⃣ Training using model.fit()
# -------------------------------
print("\n⚙️ Training using model.fit()")

model_fit = create_model()
model_fit.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Train the model
history = model_fit.fit(
    x_train, y_train_onehot,
    validation_data=(x_test, y_test_onehot),
    batch_size=64,
    epochs=5,
    verbose=2
)

# Evaluate
fit_loss, fit_accuracy = model_fit.evaluate(x_test, y_test_onehot, verbose=0)
print(f"\n✅ Test Accuracy (model.fit): {fit_accuracy:.4f}")


# -------------------------------
# 📊 Comparison
# -------------------------------
print("\n📈 Accuracy Comparison")
print("-" * 30)
print(f"GradientTape Accuracy : {gradient_tape_accuracy:.4f}")
print(f"model.fit() Accuracy  : {fit_accuracy:.4f}")


🔁 Training using tf.GradientTape

Epoch 1/5
Step 0, Loss: 2.3543
Step 200, Loss: 0.5770
Step 400, Loss: 0.1907
Step 600, Loss: 0.1476
Step 800, Loss: 0.1418
Training Accuracy: 0.9213

Epoch 2/5
Step 0, Loss: 0.0806
Step 200, Loss: 0.0893
Step 400, Loss: 0.0378
Step 600, Loss: 0.2413
Step 800, Loss: 0.0326
Training Accuracy: 0.9672

Epoch 3/5
Step 0, Loss: 0.0820
Step 200, Loss: 0.1053
Step 400, Loss: 0.0866
Step 600, Loss: 0.0288
Step 800, Loss: 0.1397
Training Accuracy: 0.9762

Epoch 4/5
Step 0, Loss: 0.0308
Step 200, Loss: 0.0223
Step 400, Loss: 0.1475
Step 600, Loss: 0.1367
Step 800, Loss: 0.2313
Training Accuracy: 0.9823

Epoch 5/5
Step 0, Loss: 0.1411
Step 200, Loss: 0.0092
Step 400, Loss: 0.0241
Step 600, Loss: 0.0265
Step 800, Loss: 0.0118
Training Accuracy: 0.9871

✅ Test Accuracy (GradientTape): 0.9740

⚙️ Training using model.fit()
Epoch 1/5
938/938 - 5s - 6ms/step - accuracy: 0.9227 - loss: 0.2696 - val_accuracy: 0.9596 - val_loss: 0.1338
Epoch 2/5
938/938 - 5s - 6ms/step -